# FYP compute benchmark on a dedicated GPU (Google Colab)

Runs the same `scripts/benchmark.py` as on the laptop, on Colab's NVIDIA GPU, in three setups:

| Setup | YOLO | DeepSORT | Face recognition |
|---|---|---|---|
| CPU only | CPU | CPU | CPU |
| CPU + GPU | GPU | CPU | CPU |
| GPU | GPU | GPU | GPU |

**Before you start**
1. In Google Drive, make a folder called `FYP_colab` and put two files in it: `colab_bench.zip` and `home_bench_cam1.mp4`.
2. In this notebook: **Runtime → Change runtime type → T4 GPU → Save**.
3. Then **Runtime → Run all**. It takes roughly 30–60 minutes; the CPU runs are the slow part.


## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Get the files from Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/FYP_colab'   # change if you used another folder name

!rm -rf /content/colab_bench
!unzip -q "$DRIVE_DIR/colab_bench.zip" -d /content
!cp "$DRIVE_DIR/home_bench_cam1.mp4" /content/colab_bench/recordings/
%cd /content/colab_bench
!ls -R | head -40

## 3. Install the libraries (2–5 minutes)

In [ ]:
!pip -q install ultralytics "deep-sort-realtime==1.3.2" "deepface==0.0.93" mtcnn loguru psutil "setuptools<81"
import subprocess
import sys

import tensorflow as tf

mm = '.'.join(tf.__version__.split('.')[:2])
print('TensorFlow', tf.__version__, '-> installing tf-keras', mm)
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', f'tf-keras=={mm}.*'], check=False)

## 4. Check that PyTorch and TensorFlow can see the GPU

In [ ]:
import tensorflow as tf
import torch

print('PyTorch CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('TensorFlow GPUs:', tf.config.list_physical_devices('GPU'))

## 5. Download the face model once (so the download is not timed)

In [ ]:
!python -c "from deepface import DeepFace; DeepFace.build_model('VGG-Face'); print('VGG-Face ready')"
!rm -f known_faces/*.pkl

## 6. Run the six benchmarks
Each run prints a report and saves it in `output/`. The file names say which setup it was (`cpu`, `cuda_hybrid`, `cuda_gpu`).

In [ ]:
COMMON = '--source recordings/home_bench_cam1.mp4 --loop --frames 1000 --imgsz 640 --weights models/yolov8n.pt'

# CPU only (GPU hidden from everything)
!CUDA_VISIBLE_DEVICES= python scripts/benchmark.py $COMMON --device cpu --no-face
!CUDA_VISIBLE_DEVICES= python scripts/benchmark.py $COMMON --device cpu

In [ ]:
# CPU + GPU: YOLO on the GPU, DeepSORT and face recognition on the CPU
!python scripts/benchmark.py $COMMON --device cuda --track-device cpu --face-device cpu --no-face
!python scripts/benchmark.py $COMMON --device cuda --track-device cpu --face-device cpu

In [ ]:
# GPU: YOLO, DeepSORT and face recognition all on the GPU
!python scripts/benchmark.py $COMMON --device cuda --no-face
!python scripts/benchmark.py $COMMON --device cuda

## 7. Summary table

In [ ]:
import glob
import json
import os

rows = []
for f in sorted(glob.glob('output/benchmark_*_summary.json')):
    d = json.load(open(f)); s = d['stats']; st = d['settings']; p = d['projection']
    rows.append([os.path.basename(f).replace('_summary.json',''), st.get('run_label'), st['face_enabled'],
                 round(s['detect']['mean'],1), round(s['track']['mean'],1),
                 round(s['face_per_call']['mean'],1), round(p['per_camera_ms_per_s']['total'],1), p['cameras_per_machine']])
print(f"{'run':45s} {'setup':12s} {'face':5s} {'YOLO':>7s} {'track':>7s} {'face/call':>9s} {'load':>7s} {'cams':>5s}")
for r in rows:
    print(f"{r[0]:45s} {str(r[1]):12s} {str(r[2]):5s} {r[3]:7.1f} {r[4]:7.1f} {r[5]:9.1f} {r[6]:7.1f} {r[7]:5d}")
print()
print(open(sorted(glob.glob('output/benchmark_*_report.md'))[-1]).read()[:600])

## 8. Save the results (to Drive and as a download)

In [ ]:
!cd /content/colab_bench && zip -q -r colab_results.zip output
!cp /content/colab_bench/colab_results.zip "$DRIVE_DIR/"
from google.colab import files

files.download('/content/colab_bench/colab_results.zip')
print('Saved colab_results.zip to', DRIVE_DIR)